In [2]:
import argparse
import os
import glob
import numpy as np
from pathlib import Path
from sklearn.model_selection import train_test_split
import tensorflow as tf
from tqdm import tqdm

In [3]:
def safe_load_npz(path):
    """Load first array from .npz and return it as numpy array."""
    data = np.load(path)
    # prefer named keys if present
    if hasattr(data, 'files') and data.files:
        key = data.files[0]
        arr = data[key]
    else:
        # fallback - try arr_0
        arr = data['arr_0'] if 'arr_0' in data else None
    data.close()
    return arr

In [4]:
def load_data(data_dir):
    data_dir = Path(data_dir)
    dirs = [data_dir / '0', data_dir / '1']
    X_list = []
    y_list = []

    for label, d in tqdm(enumerate(dirs)):
        if not d.exists():
            print(f"Warning: folder {d} does not exist, skipping")
            continue
        files = sorted(glob.glob(str(d / '*.npz')))
        if not files:
            print(f"Warning: no .npz files found in {d}")
        for f in tqdm(files):
            try:
                arr = safe_load_npz(f)
            except Exception as e:
                print(f"Failed to load {f}: {e}")
                continue
            if arr is None:
                print(f"No array found in {f}, skipping")
                continue
            # Ensure float32
            arr = arr.astype('float32')

            X_list.append(arr)
            y_list.append(label)
    X = np.array(X_list)
    Y = np.array(y_list)
    return X, Y

In [5]:
def build_simple_cnn(input_shape):
    import tensorflow as tf
    from tensorflow.keras import layers, models

    model = models.Sequential([
        layers.Input(shape=input_shape),

        layers.Conv2D(32, 3, activation='relu'),
        layers.MaxPooling2D(2),
        
        layers.Conv2D(64, 3, activation='relu'),
        layers.MaxPooling2D(2),

        layers.Conv2D(128, 3, activation='relu'),
        layers.MaxPooling2D(2),

        layers.Flatten(),

        layers.Dense(128, activation='relu'),
        layers.Dense(1, activation='sigmoid')
    ])
    model.compile(optimizer='adam',
                  loss='binary_crossentropy',
                  metrics=['accuracy', tf.keras.metrics.AUC(name='auc')])
    return model


In [ ]:
# Stratified split
X,Y = load_data("/Users/administrateur/Desktop/projects_repo/SKIPPER_collab/training_cropped_arrays")
X_train, X_val, y_train, y_val = train_test_split(
        X, Y, test_size=0.2, stratify=Y, random_state=42)

input_shape = X_train.shape[1:]
model = build_simple_cnn(input_shape)

100%|██████████| 33960/33960 [00:51<00:00, 662.09it/s]
2it [01:24, 42.37s/it]


In [ ]:
model.summary()


In [ ]:
model.fit(X_train, y_train,
              validation_data=(X_val, y_val),
              epochs=10,
              batch_size=32)

In [ ]:
from sklearn.metrics import roc_auc_score
y_pred = model.predict(X_val).ravel()
auc = roc_auc_score(y_val, y_pred)
print(f"Validation ROC-AUC: {auc:.4f}")

In [ ]:
save_path = Path(save_path)
model.save(save_path)
print(f"Model saved to {save_path}")